# Device Check

In [1]:
# ============================================================
# Project Information
# ============================================================
# Project : Vehicle Vision Assistant
# Model   : Qwen/Qwen3-VL-8B-Instruct
# Dataset : Paulescu/stanford_cars
# Method  : LoRA Fine-Tuning
# ============================================================

import platform
import torch

print("=" * 60)
print("Environment Information")
print("=" * 60)

print(f"Python          : {platform.python_version()}")
print(f"PyTorch         : {torch.__version__}")

print(f"CUDA Available  : {torch.cuda.is_available()}")
print(f"CUDA Version    : {torch.version.cuda}")

# ROCm builds report their HIP version here
print(f"ROCm Version    : {torch.version.hip}")

if torch.cuda.is_available():

    device = torch.cuda.current_device()
    props = torch.cuda.get_device_properties(device)

    print("-" * 60)
    print(f"GPU Name        : {props.name}")
    print(f"Compute Device  : {device}")
    print(f"Total VRAM      : {props.total_memory / 1024**3:.2f} GB")

print("=" * 60)

Environment Information
Python          : 3.11.9
PyTorch         : 2.11.0+cpu
CUDA Available  : False
CUDA Version    : None
ROCm Version    : None


In [2]:
!pip install trl --no-deps
!pip install peft --no-deps
!pip install -U qwen-vl-utils

# Imports

In [3]:
# Standard Library

import random

# Third-Party Libraries

import torch
import matplotlib.pyplot as plt

import datasets
import peft
import transformers
import trl

# Hugging Face

from datasets import load_dataset

from transformers import (
    AutoProcessor,
    Qwen3VLForConditionalGeneration,
)

# PEFT (LoRA)

from peft import (
    LoraConfig,
    get_peft_model,
)

# TRL

from trl import (
    SFTConfig,
    SFTTrainer,
)

# Qwen Utilities

from qwen_vl_utils import process_vision_info


SEED = 42

random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    
# Environment Information

print("=" * 60)
print("Environment")
print("=" * 60)
print(f"PyTorch       : {torch.__version__}")
print(f"Transformers  : {transformers.__version__}")
print(f"Datasets      : {datasets.__version__}")
print(f"PEFT          : {peft.__version__}")
print(f"TRL           : {trl.__version__}")
print("=" * 60)

c:\Users\user\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Environment
PyTorch       : 2.11.0+cpu
Transformers  : 5.14.1
Datasets      : 5.0.0
PEFT          : 0.19.1
TRL           : 1.9.0


# Load Dataset


In [ ]:
# Load Stanford Cars Dataset

dataset = load_dataset("Paulescu/stanford_cars")

print("-" * 60)
print("Dataset Summary")
print("-" * 60)

display(dataset)

print("\nDataset Features")
print("-" * 60)

display(dataset["train"].features)

print("\nNumber of Samples")
print("-" * 60)

print(f"Train : {len(dataset['train']):,}")
print(f"Test  : {len(dataset['test']):,}")

print("=" * 60)

# Show Sample

In [ ]:
# Display Sample from Training Dataset

sample = dataset["train"][0]

print("-" * 60)
print("Sample Information")
print("-" * 60)

print(f"Maker : {sample['maker']}")
print(f"Model : {sample['model']}")
print(f"Year  : {sample['year']}")

print("-" * 60)


plt.figure(figsize=(8, 8))
plt.imshow(sample["image"])
plt.title(f"{sample['maker']} | {sample['model']} ({sample['year']})")
plt.axis("off")
plt.show()

# Dataset Statistics

In [ ]:
train_size = len(dataset["train"])
test_size = len(dataset["test"])

makers = sorted(set(dataset["train"]["maker"]))
models = sorted(set(dataset["train"]["model"]))
years = sorted(set(dataset["train"]["year"]))

print("=" * 60)
print("Dataset Statistics")
print("=" * 60)

print(f"Training Samples : {train_size:,}")
print(f"Testing Samples  : {test_size:,}")
print(f"Manufacturers    : {len(makers)}")
print(f"Vehicle Models   : {len(models)}")
print(f"Production Years : {len(years)}")

print("-" * 60)
print(f"Years : {years}")

print("-" * 60)
print("Manufacturers:")
print(makers)

print("=" * 60)

In [ ]:
# ============================================================
# Dataset Labels
# ============================================================

label_names = dataset["train"].features["label"].names

print("=" * 60)
print("Dataset Labels")
print("=" * 60)
print(f"Number of Classes : {len(label_names)}")
print(f"First Label       : {label_names[0]}")
print("=" * 60)

# Load Model

In [ ]:
# Configuration

MODEL_NAME = "Qwen/Qwen3-VL-8B-Instruct"

DTYPE = torch.bfloat16
DEVICE_MAP = "auto"

print("-" * 60)
print("Model Configuration")
print("-" * 60)
print(f"Model      : {MODEL_NAME}")
print(f"Data Type  : {DTYPE}")
print(f"Device Map : {DEVICE_MAP}")
print("-" * 60)

In [ ]:
# Load Processor & Model

processor = AutoProcessor.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
)

model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=DTYPE,
    device_map=DEVICE_MAP,
    trust_remote_code=True,
)

print("-" * 60)
print("Model Loaded Successfully")
print("-" * 60)
print(f"Model Type      : {model.config.model_type}")
print(f"Vocabulary Size : {processor.tokenizer.vocab_size:,}")
print("-" * 60)

In [ ]:
# Training Prompts

PROMPTS = [
    "Identify the vehicle in this image.",
    "What car is shown in the image?",
    "Recognize the manufacturer, model, and year of this vehicle.",
    "Identify this vehicle as accurately as possible.",
]

print("-" * 60)
print("Training Prompts")
print("-" * 60)

for i, prompt in enumerate(PROMPTS, start=1):
    print(f"{i}. {prompt}")

print("-" * 60)

# LoRA Config

In [ ]:
# ============================================================
# LoRA Configuration
# ============================================================

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

print("-" * 60)
print("LoRA Configuration")
print("-" * 60)
print(f"Rank (r)        : {lora_config.r}")
print(f"LoRA Alpha      : {lora_config.lora_alpha}")
print(f"LoRA Dropout    : {lora_config.lora_dropout}")
print(f"Target Modules  : {len(lora_config.target_modules)}")
print("-" * 60)

In [ ]:
# ============================================================
# Apply LoRA
# ============================================================

model = get_peft_model(model, lora_config)

print("=" * 60)
print("LoRA Applied Successfully")
print("=" * 60)

model.print_trainable_parameters()

print("=" * 60)

# SFTConfig

In [ ]:
# Training Configuration

OUTPUT_DIR = "./qwen3vl-car-lora"

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=10,
    save_strategy="epoch",
    remove_unused_columns=False,
    report_to="none",
)

print("=" * 60)
print("Training Configuration")
print("=" * 60)
print(f"Epochs             : {training_args.num_train_epochs}")
print(f"Batch Size         : {training_args.per_device_train_batch_size}")
print(f"Gradient Accum     : {training_args.gradient_accumulation_steps}")
print(f"Learning Rate      : {training_args.learning_rate}")
print(f"Output Directory   : {training_args.output_dir}")
print("=" * 60)

In [ ]:
# Data Collator

def collate_fn(examples):

    texts = []
    images = []

    for example in examples:

        prompt = random.choice(PROMPTS)
        answer = label_names[example["label"]]

        conversation = [
            {
                "role": "user",
                "content": [
                    {
                        "type": "image",
                        "image": example["image"],
                    },
                    {
                        "type": "text",
                        "text": prompt,
                    },
                ],
            },
            {
                "role": "assistant",
                "content": [
                    {
                        "type": "text",
                        "text": answer,
                    }
                ],
            },
        ]

        text = processor.apply_chat_template(
            conversation,
            tokenize=False,
            add_generation_prompt=False,
        )

        image_inputs, _ = process_vision_info(conversation)

        texts.append(text)
        images.append(image_inputs)

    batch = processor(
        text=texts,
        images=images,
        padding=True,
        return_tensors="pt",
    )

    labels = batch["input_ids"].clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100

    batch["labels"] = labels

    return batch

print("✓ Custom data collator is ready.")

# Trainer

In [ ]:
# Initialize SFT Trainer

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    processing_class=processor,
    data_collator=collate_fn,
)

print("=" * 60)
print("SFTTrainer Initialized Successfully")
print("=" * 60)
print(f"Training Samples : {len(dataset['train']):,}")
print(f"Training Epochs  : {training_args.num_train_epochs}")
print("=" * 60)

In [ ]:
# ============================================================
# Verify Data Collator
# ============================================================

batch = collate_fn([dataset["train"][0]])

print("=" * 60)
print("Batch Information")
print("=" * 60)

for key, value in batch.items():
    if isinstance(value, torch.Tensor):
        print(f"{key:<20} {tuple(value.shape)}")
    else:
        print(f"{key:<20} {type(value)}")

print("=" * 60)

In [ ]:
#cheack for divice 
!rocm-smi

# Start Fine-Tuning

In [ ]:
# Start Fine-Tuning

trainer.train()
print(trainer.state)

# Save LoRA Adapter

In [ ]:
# ============================================================
# Save LoRA Adapter
# ============================================================

trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)

print("=" * 60)
print("LoRA Adapter Saved Successfully")
print(f"Path : {OUTPUT_DIR}")
print("=" * 60)

In [ ]:
# ============================================================
# Select Test Samples
# ============================================================

NUM_SAMPLES = 5

test_samples = random.sample(
    range(len(dataset["test"])),
    NUM_SAMPLES,
)

print("=" * 60)
print(f"Testing on {NUM_SAMPLES} Random Images")
print("=" * 60)

In [ ]:
# ============================================================
# Text Normalization
# ============================================================

import re

def normalize_text(text):
    """
    Normalize text before comparison.
    """

    text = text.lower().strip()

    # Remove punctuation
    text = re.sub(r"[^\w\s]", "", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text)

    return text

In [ ]:
# ============================================================
# Test Fine-Tuned Model
# ============================================================

correct = 0
manufacturer_score = 0
model_score = 0
year_score = 0
overall_score = 0
for i, idx in enumerate(test_samples, start=1):

    sample = dataset["test"][idx]

    image = sample["image"]
    ground_truth = label_names[sample["label"]]

    conversation = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": image,
                },
                {
                    "type": "text",
                    "text": "Identify the manufacturer, model, and year of this vehicle.",
                },
            ],
        }
    ]

    text = processor.apply_chat_template(
        conversation,
        tokenize=False,
        add_generation_prompt=True,
    )

    image_inputs, _ = process_vision_info(conversation)

    inputs = processor(
        text=[text],
        images=image_inputs,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=30,
        )

    generated_ids = output[:, inputs.input_ids.shape[1]:]

    prediction = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True,
    )[0].strip()

    prediction_clean = normalize_text(prediction)
    ground_truth_clean = normalize_text(ground_truth)
    gt_manufacturer, gt_model, gt_year = parse_vehicle(ground_truth)

    pred_manufacturer, pred_model, pred_year = parse_vehicle(prediction)

    manufacturer_correct = pred_manufacturer == gt_manufacturer
    model_correct = pred_model == gt_model
    year_correct = pred_year == gt_year

    sample_correct = (
        manufacturer_correct
        and model_correct
        and year_correct
    )

    manufacturer_score += manufacturer_correct
    model_score += model_correct
    year_score += year_correct
    overall_score += sample_correct

    is_correct = prediction_clean == ground_truth_clean

    if is_correct:
        correct += 1

    plt.figure(figsize=(6, 6))
    plt.imshow(image)
    plt.axis("off")
    plt.title(f"Sample {i}")
    plt.show()

    print("=" * 60)
    print(f"Ground Truth : {ground_truth}")
    print(f"Prediction   : {prediction}")
    print(f"Correct      : {is_correct}")
    print("=" * 60)
    print("=" * 60)
    print(f"Ground Truth : {ground_truth}")
    print(f"Prediction   : {prediction}")
    plt.figure(figsize=(6, 6))
    plt.imshow(image)
    plt.axis("off")
    plt.show()

    print("=" * 60)
    print(f"Ground Truth : {ground_truth}")
    print(f"Prediction   : {prediction}")

    print(f"Manufacturer : {'PASS' if manufacturer_correct else 'FAIL'}")
    print(f"Model        : {'PASS' if model_correct else 'FAIL'}")
    print(f"Year         : {'PASS' if year_correct else 'FAIL'}")

    print(f"Overall      : {'PASS' if sample_correct else 'FAIL'}")
    print("=" * 60)

    if is_correct:
        print("Result       : PASS")
    else:
        print("Result       : FAIL")

    print("=" * 60)

In [ ]:
# ============================================================
# Inference Summary
# ============================================================

accuracy = correct / NUM_SAMPLES * 100

print("=" * 60)
print("Inference Summary")
print("=" * 60)
print(f"Correct Predictions : {correct}/{NUM_SAMPLES}")
print(f"Accuracy            : {accuracy:.2f}%")
print("=" * 60)